# v2 Notebook 02: Prepare all five days

**Author:** Anush Jindal

v1 used one file (Wednesday). v2 uses all eight CICIDS2017 files, Monday to Friday, so that
a tree can be tested on days and attacks it has never seen.

This notebook:

1. cleans each file exactly like v1 (infinite values, missing values, exact duplicates);
2. keeps each attack's name, but adds a simple **attack / normal** label, which is what v2's
   trees learn;
3. marks every flow as **train**, **validation** or **test** by time order;
4. saves one table, `data/all_days_clean.pkl`, for the other notebooks.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))  # finds helpers.py

import numpy as np
import pandas as pd
import helpers as h

h.RESULTS_DIR.mkdir(exist_ok=True)

## Clean each file

In [2]:
parts, summary = [], []
for day, filename in h.DAY_FILES:
    df, counts = h.load_clean(h.DATA_DIR / filename)
    df.insert(0, "file", filename)
    df.insert(0, "day", day)
    parts.append(df)
    summary.append({"day": day, "file": filename, **counts,
                    "normal_flows": int((df["Label"] == h.BENIGN).sum()),
                    "attack_flows": int((df["Label"] != h.BENIGN).sum())})

data = pd.concat(parts, ignore_index=True)
summary = pd.DataFrame(summary)
summary.loc[len(summary)] = {"day": "All", "file": "(8 files)", **summary.drop(columns=["day", "file"]).sum().to_dict()}
summary.to_csv(h.RESULTS_DIR / "02_data_summary.csv", index=False)
summary

Monday-WorkingHours.pcap_ISCX.csv: 529,918 -> 502,650 rows


Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 -> 421,626 rows


Wednesday-workingHours.pcap_ISCX.csv: 692,703 -> 610,492 rows


Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 -> 164,179 rows


Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 -> 252,790 rows


Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 -> 184,044 rows


Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 -> 213,777 rows


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 -> 223,082 rows


,day,file,rows_raw,infinite_removed,missing_removed,duplicates_removed,rows_clean,normal_flows,attack_flows
0,Monday,Monday-WorkingHours.pcap_ISCX.csv,529918,437,0,26831,502650,502650,0
1,Tuesday,Tuesday-WorkingHours.pcap_ISCX.csv,445909,264,0,24019,421626,412476,9150
2,Wednesday,Wednesday-workingHours.pcap_ISCX.csv,692703,1297,0,80914,610492,416736,193756
3,Thursday,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,135,0,6052,164179,162036,2143
4,Thursday,Thursday-WorkingHours-Afternoon-Infilteration....,288602,207,0,35605,252790,252754,36
5,Friday,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,122,0,6867,184044,182096,1948
6,Friday,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,371,0,72319,213777,123083,90694
7,Friday,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,34,0,2629,223082,95068,128014
8,All,(8 files),2830743,2867,0,255236,2572640,2146899,425741


## Mark train / validation / test by time order

Within each file, each kind of traffic (normal, or one attack) is cut by position: the
**first 60%** is `train`, the **next 20%** is `validation` and the **last 20%** is `test`.
The files list flows in the order the traffic was recorded, so "later rows" means "later in
the day". (The public CSV files have no timestamp column, so file order is the best
available clock.)

In [3]:
data.insert(2, "part", h.time_parts(data))
data["Label"].where(data["Label"] == h.BENIGN, "attack").groupby(data["part"]).value_counts().unstack()

Label,BENIGN,attack
part,,
test,429377,85143
train,1288142,255450
validation,429380,85148


In [4]:
attacks = (data[data["Label"] != h.BENIGN]
           .groupby(["day", "Label", "part"], sort=False).size().unstack("part")[["train", "validation", "test"]])
attacks["total"] = attacks.sum(axis=1)
attacks = attacks.reset_index().rename(columns={"Label": "attack"})
attacks.to_csv(h.RESULTS_DIR / "02_attack_counts.csv", index=False)
attacks

part,day,attack,train,validation,test,total
0,Tuesday,FTP-Patator,3559,1186,1186,5931
1,Tuesday,SSH-Patator,1932,644,643,3219
2,Wednesday,DoS slowloris,3231,1077,1077,5385
3,Wednesday,DoS Slowhttptest,3137,1046,1045,5228
4,Wednesday,DoS Hulk,103708,34569,34569,172846
5,Wednesday,DoS GoldenEye,6172,2057,2057,10286
6,Wednesday,Heartbleed,7,2,2,11
7,Thursday,Web Attack - Brute Force,882,294,294,1470
8,Thursday,Web Attack - XSS,392,130,130,652
9,Thursday,Web Attack - Sql Injection,13,4,4,21


## Save

The 78 feature columns are stored as 32-bit numbers. scikit-learn converts every feature to
32-bit before training a tree anyway, so this changes no result and halves the file size.

In [5]:
feature_columns = [c for c in data.columns if c not in ("day", "file", "part", "Label")]
assert len(feature_columns) == 78
data[feature_columns] = data[feature_columns].astype(np.float32)
data.to_pickle(h.CLEAN_FILE)
print(f"Saved {len(data):,} flows x {len(feature_columns)} features to {h.CLEAN_FILE.name}")
print(f"Attack types: {data['Label'].nunique() - 1}, days: {data['day'].nunique()}")

Saved 2,572,640 flows x 78 features to all_days_clean.pkl
Attack types: 14, days: 5
